# Решения: permutation practice

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

## Карта эталона

**Фокус:** Воспроизводимая функция перестановочного теста.

Собираем расчёт в функцию с явными параметрами `n_iter` и `seed`, проверяем её на сегментах и отделяем величину эффекта от статистической совместимости с H0.

Эталон разделён на исполняемые секции в том же порядке, что `lesson.ipynb` и `homework.ipynb`. После каждой секции сверяйте не только значение, но и способ вычисления.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_06_ab_startup/data/" + name


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


## 0.1. Контракт перестановочного теста

До функции зафиксируйте estimand, альтернативу, число симуляций и seed. Контракт не должен зависеть от результата конкретного запуска.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
test_contract = {
    'estimand': 'conv_B - conv_A',
    'alternative': 'two-sided',
    'n_iter': 3000,
    'seed': 37,
}
assert test_contract['alternative'] == 'two-sided'

## 0.2. Проверка входных данных

Докажите, что обе группы присутствуют, target бинарный, а пропуски не попадут в симуляцию. Сохраните результат в компактной `Series`.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
data_check = pd.Series({
    'n_rows': len(df),
    'n_variants': df['variant'].nunique(),
    'missing_target': df['converted'].isna().sum(),
    'binary_target': set(df['converted'].unique()) <= {0, 1},
})
assert bool(data_check['binary_target'])

## 0.3. План проверки воспроизводимости

Запишите две пары настроек: одинаковые seed должны повторить результат, разные seed могут дать небольшой Monte Carlo-разброс. Здесь мы проверяем протокол, а не ищем удобное p-value.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
repro_seeds = (37, 37, 38)
REPRO_NOTE = (
    'Два запуска с seed=37 должны совпасть; seed=38 оценивает допустимый Monte Carlo-шум, '
    'но не используется для выбора более удобного результата.'
)
assert len(REPRO_NOTE) > 90

## Решение 1

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
def permutation_test_diff(frame, n_iter=3000, seed=0):
    rng = np.random.default_rng(seed)
    conv = frame['converted'].to_numpy()
    mask_b = frame['variant'].to_numpy() == 'B'
    obs = float(conv[mask_b].mean() - conv[~mask_b].mean())
    sims = np.empty(n_iter)
    for i in range(n_iter):
        perm = rng.permutation(conv)
        sims[i] = perm[mask_b].mean() - perm[~mask_b].mean()
    p = float((np.abs(sims) >= abs(obs)).mean())
    return obs, p

## Решение 2

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
obs, p = permutation_test_diff(df, n_iter=3000, seed=37)

mob = df[df['device'] == 'mobile']

des = df[df['device'] == 'desktop']

## Решение 3

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
obs_mob, p_mob = permutation_test_diff(mob, n_iter=3000, seed=38)

obs_des, p_des = permutation_test_diff(des, n_iter=3000, seed=39)

_, p_500 = permutation_test_diff(df, n_iter=500, seed=40)

## Решение 4

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
_, p_2000 = permutation_test_diff(df, n_iter=2000, seed=40)

_, p_8000 = permutation_test_diff(df, n_iter=8000, seed=40)

INTERP = (
    'p-value отвечает на вопрос совместимости данных с H0, а не на вопрос масштаба эффекта. '
    'Всегда интерпретируем p-value вместе с самой разницей конверсий и бизнес-контекстом.'
)

## Решение 5

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
ads = df[df['traffic_source'] == 'ads']

_, p_ads = permutation_test_diff(ads, n_iter=3000, seed=41)

rng = np.random.default_rng(77)

## Решение 6

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
half_idx = rng.choice(df.index.to_numpy(), size=len(df) // 2, replace=False)

half = df.loc[half_idx]

_, p_half = permutation_test_diff(half, n_iter=3000, seed=42)

## Решение 7

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
rng2 = np.random.default_rng(43)

conv = df['converted'].to_numpy()

mask_b = df['variant'].to_numpy() == 'B'

## Решение 8

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
sim_vals = []

for _ in range(1200):
    perm = rng2.permutation(conv)
    sim_vals.append(float(perm[mask_b].mean() - perm[~mask_b].mean()))

## Решение 9

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
sim_table = pd.DataFrame({'sim_diff': sim_vals})

PHACK_NOTE = (
    'Если запускать тест много раз, отбирать удобные подвыборки и останавливать анализ на удачном моменте, '
    'можно получить ложную значимость даже без реального эффекта.'
)

print(round(p, 5), round(p_mob, 5), round(p_des, 5), round(p_ads, 5), round(p_half, 5))

## Проверка преподавателя

Запустите `Run All`. Эталон должен завершиться без исключений; итоговые числа должны совпадать при повторном запуске благодаря фиксированным seed. Текстовый вывод проверяется на согласованность с направлением uplift, p-value и границами CI.